<a href="https://colab.research.google.com/github/abrham17/ML_Pytorch_repo/blob/main/trandfomer_implementation_from_the_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class Embedding(nn.Module):
  def __init__(self , vocab_size : int , d_model:int):
    super().__init__()
    self.vocab_size = vocab_size
    self.d_model = d_model
    self.embed = nn.Embedding(vocab_size , d_model)
  def forward(self , x):
    return self.embed(x) * math.sqrt(self.d_model)


In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self , d_model:int , seq_len:int , dropout:float):
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)
    pe = torch.zeros(seq_len , d_model)
    divterm = torch.exp(torch.arange(0 , d_model , 2) * -(math.log(10000.0)/d_model))
    pe[:, 0::2] = torch.sin(torch.arange(seq_len).unsqueeze(1) * divterm)
    self.pe[:, 1::2] = torch.cos(torch.arange(seq_len).unsqueeze(1) * divterm)
  def forward(self , x):
    self.pe = x + self.pe[:x.size(0) , :].requires_grad_(false)
    self.pe = self.dropout(self.pe)
    return self.pe

In [ ]:
class LayerNormalization(nn.Module):
  def __init__(self , epsilon:float):
    super().__init__()
    self.epsilon = epsilon
    self.alpha = nn.Parameter(torch.ones(1))
    self.bias = nn.Parameter(torch.zeros(1))
  def forward(self,x):
    x = self.alpha * (x - torch.mean(x , dim = -1 , keepdim = True)) / (torch.std(x , dim = -1 , keepdim = True) + self.epsilon) + self.bias
    return x

In [ ]:
class FeedForwardLayer(nn.Module):
  def __init__(self, d_model:int , dff:int , dropout:float):
    super().__init__()
    self.l1 = nn.Linear(d_model , dff)
    self.dropout = nn.Dropout(dropout)
    self.l2 = nn.Linear(dff , d_model)
  def forward(self , x):
    return self.l2(self.dropout(torch.relu(self.l1(x))))

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self , d_model:int , h:int):
    super().__init__()
    self.d_model = d_model
    assert d_model % h == 0 , "d_model must be divisible by h"
    self.h = h
    self.d_k = d_model // h
    self.wk = nn.Linear(d_model , d_model)
    self.wq = nn.Linear(d_model , d_model)
    self.wv = nn.Linear(d_model , d_model)

    self.wo = nn.Linear(d_model , d_model)
    self.attention_scores = None

  @staticmethod
  def attention(query , key , value , mask):
    scores = torch.matmul(query , key.transpose(-2 , -1)) / math.sqrt(query.shape[-1])
    if mask is not None:
      scores = scores.masked_fill(mask == 0 , -1e9)
    scores = torch.softmax(scores , dim = -1)
    return torch.matmul(scores , value) , scores

  def forward(self , q , k , v , mask):
    self.q = self.wq(q)
    self.k = self.wk(k)
    self.v = self.wv(v)
    query = self.q.view(self.q.shape[0], self.q.shape[1] , self.h , self.d_k).transpose(2 , 1)
    key = self.k.view(self.k.shape[0] , self.q.shape[1] , self.h , self.d_k).transpose(2 , 1)
    value = self.v.view(self.v.shape[0] ,self.q.shape[1] , self.h , self.d_k).transpose(2 , 1)
    x , self.attention_scores = MultiHeadAttention.attention(query , key , value , mask)
    x = x.transpose(2 , 1).contiguous().view(x.shape[0] , x.shape[1] , self.d_model)
    return self.wo(x)
#


In [ ]:
class ResidualConnection(nn.Module):
  def __init__(self , dropout:float):
    super().__init__()
    self.dropout = nn.Dropout(dropout)

  def forward(self , x , sublayer):
    return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self , attention_block:MultiHeadAttention , FFN:FeedForwardLayer , dropout:float) -> None:
    super().__init__()
    self.attention_block = attention_block
    self.FFN = self.FFN
    self.dropout = dropout
    self.residual_layer = nn.ModuleList([ResidualConnection(dropout) for i in range(2)])

  def forward(self , x , mask):
    x = self.residual_layer[0](x , lambda x:self.attention_block(x , x , x , mask))
    x = self.residual_layer[1](x , lambda x:self.FFN(x))
    return x


In [ ]:
class Encoder(nn.Module):
  def __init__(self , layer:nn.ModuleList) -> None:
    super().__init__()
    self.layers = layers
    self.norm = LayerNormalization(1e-6)
  def forward(self , x):
    for layer in self.layers:
      x = layer(x)
    return self.norm(x)


In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self , musked_attention:MultiHeadAttention , attention2:MultiHeadAttention , FFN:FeedForwardLayer , cross_attention:MultiHeadAttention , dropout:float) -> None:
    super().__init__()
    self.musked_attention = attention
    self.cross_attention = cross_attention
    self.attention2 = attention2
    self.FFN = FFN
    self.dropout = dropout
    self.residual_connection = nn.ModuleList([ResidualConnections(dropout) for _ in range(3)])

  def forward(self , x , encoder_output , enc_mask, tar_mark):
    x = self.residual_connection[0](x , lambda x: self.musked_attention(x , x , x , tar_mask))
    x = self.residual_connection[1](x , lambda x: self.musked_attention(encoder_output , encoder_output , x , enc_mask))
    x = self.residual_connection[2](x , lambda x: self.FFN(x))
    return x

In [ ]:
class Decoder(nn.Module):
  def __init__(self, layer:nn.ModuleList) -> None:
    super().__init__()
    self.layers = layers
    self.norm = LayerNormalization(1e-6)
  def forward(self , x , encoder_output , enc_mask , tar_mask):
    for layer in self.layers:
      x = layer(x , encoder_output , enc_mask , tar_mask)
    return self.norm(x)

In [ ]:
class LinearLayer(nn.Module):
  def __init__(self , d_model:int , vocab_size:int) -> None:
    super().__init__()
    self.linear = nn.Linear(d_model , vocab_size)
  def forward(self , x):
    return self.log_softmax(self.linear(x) , dim=-1)

In [ ]:
class Transformer(nn.Module):
  def __init__(self , encoder:Encoder , decoder:Decoder , linear_layer:LinearLayer , src_embed:InputEmbedding , trg_embed:InputEmbedding , src_pos : PositionalEncoding , tar_pos:PositionalEncoding) -> None:
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.linear_layer = linear_layer
    self.src_embed = src_embed
    self.trg_embed = trg_embed
    self.src_pos = src_pos
    self.tar_pos = tar_pos
  def encoder(self , src , src_mask):
    src = self.src_embed(src , src_mask)
    src = self.src_pos(src)
    return self.encoder(src , src_mask)
  def decoder(self , tar , tar_mask ):
    tar = self.trg_embed(tar , tar_mask)
    tar = self.tar_pos(tar)
    tar = self.decoder(tar , self.encoder ,enc_mask ,  tar_mask)
    return tar
  def projection_layer(self , x):
    return self.linear_layer(x)
  def forward(self , x):


In [ ]:
def build_transformer(src_vocab_size:int , trg_vocab_size:int , src_seq_len:int , trg_seq_len:int , src , trg , d_model:int = 512 , N:int = 6 , h:int = 8 , dropout:float = 0.1 , dff:int = 2048) -> Transformer:
  encoder = Encoder(nn.ModuleList([EncoderBlock(MultiHeadAttention(d_model , h) , FeedForwardLayer(d_model , dff , dropout)) for _ in range(N)]))
  decoder = Decoder(nn.ModuleList([DecoderBlock(MultiHeadAttention(d_model , h) , MultiHeadAttention(d_model , h) , FeedForwardLayer(d_model , dff , dropout) , MultiHeadAttention(d_model , h)) for _ in range(N)]))
  projection_layer = LinearLayer(d_model , trg_vocab_size)
  src_embed = Embedding(src_vocab_size , d_model)
  trg_embed = Embedding(trg_vocab_size , d_model)
  src_pos = PositionalEncoding(d_model , src_seq_len , dropout)
  trg_pos = PositionalEncoding(d_model , trg_seq_len , dropout)

  transformer = Transformer(encoder , decoder , projection_layer , src_embed , trg_embed , src_pos , trg_pos)

  return transformer

In [ ]:
from dataset import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainer import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace